# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RamyaMuthumariappan/FlyRankAI/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. Lane 2: Refresh / Content Opportunity Scoring

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

I'm picking this over Lane 1 because Lane 1's content-type split is roughly 90%+ single category — too imbalanced to be a real learning problem. Lane 2 gives me an honest ranking task instead: which of ~30k pages a content editor should look at first. It also forces good habits early, because the columns that look like the most obvious features (`trend_direction`, `trend_pct`) are the ones I'm not allowed to use — they're derived from the same signal I'd be trying to predict. Working that constraint out properly in week 1 sets up everything after it.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Lane: Lane 2 — Refresh / Content Opportunity Scoring")
print("Rejected: Lane 1 (content-type class imbalance ~90%+ single type)")

Lane: Lane 2 — Refresh / Content Opportunity Scoring
Rejected: Lane 1 (content-type class imbalance ~90%+ single type)


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision:** out of ~30,000 pages, which ones does the content editor open this week — and for each, is the suggested action refresh, expand, protect, prune, or monitor?

**Who acts:** the human content editor working the internal review queue. My model doesn't touch anything live — it only orders their worklist and attaches a reason code they can sanity-check in seconds.

**Cost of a wrong call is asymmetric:**
- *False positive* (page ranked high, editor opens it, nothing's actually wrong): burns a scarce editor-hour. If an editor works through ~20 pages a week and a third of my top-30 are false alarms, that's ~7 wasted editor-hours a week — the queue stops being trusted.
- *False negative* (a genuinely decaying page never surfaces): the cost is quieter but compounds — a page bleeding page-one position keeps bleeding, silently, because nobody was ever told to look.

That asymmetry is why precision@K on the top of the queue matters more to me than raw accuracy across all 30k pages — nobody works through all 30k.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
editor_pages_per_week = 20
false_positive_rate_assumed = 1/3
wasted_hours = editor_pages_per_week * false_positive_rate_assumed
print(f"Illustrative: at a {false_positive_rate_assumed:.0%} false-positive rate in a {editor_pages_per_week}-page/week queue,")
print(f"~{wasted_hours:.0f} editor-hours/week are spent on pages that didn't need attention.")



## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Three numbers from the 30,000-row starter set that make this worth doing:

1. **54.2%** of pages are currently trending down (`trend_direction == 'down'`) — more than half the catalogue, so the review problem is real, not a rare-event problem.
2. **24.4%** of pages sit at `avg_position <= 10` (page one) *and* are trending down — that's the page-one decay risk group: pages with something to protect that could be actively losing it.
3. **23.6%** of pages haven't been touched in 60+ days but still pull meaningful impressions (>50 in the last 30 days) — stale-but-visible pages, one of the clearest 'refresh first' signals.

None of this uses `trend_direction` or `trend_pct` as a *feature* — here they're just descriptive, to size the problem before I touch modeling.

In [3]:

# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Rows, columns:", df.shape)

pct_down = (df["trend_direction"] == "down").mean() * 100
print(f"Trending down: {pct_down:.1f}%")

page1_decay = ((df["avg_position"] <= 10) & (df["trend_direction"] == "down")).mean() * 100
print(f"Page-one decay risk (avg_position<=10 AND trend down): {page1_decay:.1f}%")

stale_visible = ((df["days_since_last_update"] > 60) & (df["impressions_last_30d"] > 50)).mean() * 100
print(f"Stale (>60d since update) but visible (>50 impressions/30d): {stale_visible:.1f}%")


Rows, columns: (30000, 44)
Trending down: 54.2%
Page-one decay risk (avg_position<=10 AND trend down): 24.4%
Stale (>60d since update) but visible (>50 impressions/30d): 23.6%


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can say:** this is an observed, decision-support ranking. "These 30 pages show a combination of high past visibility and negative recent movement — recommended for editor review this week, with reason codes attached." The score is directional risk, built to prioritize scarce editorial attention, not a certainty.

**What I can never say:** that a refresh *will* fix a page (correlation in the training window isn't a guarantee going forward), that I'm predicting Google's ranking algorithm, or that a high score proves causation between any single factor and decline. `avg_position` is an input I observe about where a page stands today — not something I'm forecasting.

**Leakage guard (the one rule that can't slip):** `trend_direction` and `trend_pct` are used to construct the proxy label itself, so they are never allowed in the feature set — only in descriptive stats like section 3, or in post-hoc explanation shown *after* a page is already flagged.

In [4]:
excluded_from_features = ["trend_direction", "trend_pct"]
print("Never allowed as model features (label-derived):", excluded_from_features)

candidate_features = [c for c in df.columns if c not in excluded_from_features + ["content_id", "client_id"]]
print(f"Everything else ({len(candidate_features)} columns) is a candidate feature, pending further review in w03.")


Never allowed as model features (label-derived): ['trend_direction', 'trend_pct']
Everything else (40 columns) is a candidate feature, pending further review in w03.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.